In [ ]:
import juliapkg
print("Julia exe:", juliapkg.executable())
print("Julia project:", juliapkg.project())

In [ ]:
from juliacall import Main as jl
import numpy as np
import cupy as cp
from cupyx.scipy.sparse import csr_matrix
# Load Clarabel in Julia
jl.seval('using LinearAlgebra, SparseArrays')
jl.seval('using CUDA, CUDA.CUSPARSE')

# Load Clarabel in Julia
jl.seval('using Clarabel')
# Get Python extension in Clarabel.jl
pyext = jl.Base.get_extension(jl.Clarabel, jl.Symbol("PythonExt"))
if pyext is None:
    raise RuntimeError("PythonExt extension not loaded. Ensure PythonCall is installed in the active Julia project.")


# Example: Solve a simple second-order cone programming (SOCP) problem

In [ ]:
jl.seval('''
    P = CuSparseMatrixCSR(sparse([2.0 1.0 0.0;
                1.0 2.0 0.0;
                0.0 0.0 2.0]))
    q = CuVector([0, -1., -1])
    A = CuSparseMatrixCSR(SparseMatrixCSC([1. 0 0; -1 0 0; 0 -1 0; 0 0 -1]))
    b = CuVector([1, 0., 0., 0.])

    # 0-cone dimension 1, one second-order-cone of dimension 3
    cones = Dict("f" => 1, "q"=> [3])

    settings = Clarabel.Settings(direct_solve_method = :cudss)
                                    
    solver   = Clarabel.Solver(P,q,A,b,cones, settings)
    Clarabel.solve!(solver)
    
    # Extract solution
    x = solver.solution
''')



# Reoptimization

In [ ]:
# Update b vector
bpy = cp.array([2.0, 1.0, 1.0, 1.0], dtype=cp.float64)
bjl = pyext.cupy_to_cuvector(jl.Float64, int(bpy.data.ptr), bpy.size)

# "_b" is the replacement of "!" in julia function
jl.Clarabel.update_b_b(jl.solver,bjl)          #Clarabel.update_b!()

# Update P matrix
# Define a new CSR sparse matrix on GPU
Ppy = csr_matrix(cp.array([
    [3.0, 0.5, 0.0],
    [0.5, 2.0, 0.0],
    [0.0, 0.0, 1.0]
], dtype=cp.float64))

# Extract the pointers (as integers)
data_ptr    = int(Ppy.data.data.ptr)
indices_ptr = int(Ppy.indices.data.ptr)
indptr_ptr  = int(Ppy.indptr.data.ptr)

# Get matrix shape and non-zero count
n_rows, n_cols = Ppy.shape
nnz = Ppy.nnz

jl.Pjl = pyext.cupy_to_cucsrmat(jl.Float64, data_ptr, indices_ptr, indptr_ptr, n_rows, n_cols, nnz)

jl.Clarabel.update_P_b(jl.solver, jl.Pjl)          #Clarabel.update_P!()

#Solve the new problem without creating memory
jl.Clarabel.solve_b(jl.solver)                  #Clarabel.solve!()

In [ ]:
# Retrieve the solution from Julia to Python
solution = np.array(jl.solver.solution.x)
print("Solution:", solution)

In [ ]:
print("Requesting Julia to stop...")
jl.seval("""
# global variable to signal stop request
global STOP_REQUESTED = false

function request_stop()
    global STOP_REQUESTED = true
end
""")
jl.request_stop()

# Time for Julia to process the stop request and clean up resources
import time
time.sleep(1)

print("Julia is now safe to restart kernel")